In [45]:
import pandas as pd
import numpy as np
import re

In [46]:
race = 'TOR330'

In [47]:
TOR330_itra_2018_2019_including_DNF_df = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_itra_2018_2019_including_DNFs_df.xlsx' )
TOR330_dem = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_dem.xlsx' )

In [48]:
TOR330_dem = TOR330_dem[ ['Year', 'Race', 'Name', 'Sex', 'Nationality', 'Category',
       'Status', 'Status1','Duration_seconds','Finish Category']]

In [49]:
TOR330_itra_2018_2019_including_DNF_df[['Race', 'Year', 'Name', 'ITRA_Nationality', 'Sex', 'Age', 'Performance',
       'Performance_Seconds', 'Status', 'Status1']]

,Race,Year,Name,ITRA_Nationality,Sex,Age,Performance,Performance_Seconds,Status,Status1
0,TOR330,2019,Bosatelli Oliviero,ITA,M,50,3 days 00:37:13,261433.0,Finished,Finished
1,TOR330,2019,Reynolds Galen,CAN,M,35,3 days 05:06:12,277572.0,Finished,Finished
2,TOR330,2019,Lantermino Danilo,ITA,M,38,3 days 07:09:46,284986.0,Finished,Finished
3,TOR330,2019,Erwee Tiaan,RSA,M,32,3 days 08:18:24,289104.0,Finished,Finished
4,TOR330,2019,Lukas Jens,GER,M,53,3 days 13:04:04,306244.0,Finished,Finished
...,...,...,...,...,...,...,...,...,...,...
1815,TOR330,2018,Zdon Bill,USA,M,33,NaT,NaN,DNF,DNF
1816,TOR330,2018,Zennaro Davide,ITA,M,57,NaT,NaN,DNF,DNF
1817,TOR330,2018,Zimei Andrea,ITA,M,38,NaT,NaN,DNF,DNF
1818,TOR330,2018,Zimmermann Denise,SUI,F,43,NaT,NaN,DNF,DNF


In [50]:

# Create a new column 'Finish Category'
def categorize_duration(hours):
    if hours < 60:
        return 'Sub-60'
    elif hours <= 150:
        return f'{int(hours // 10) * 10}-{int(hours // 10) * 10+9}'  # Round to nearest 10 up to 150
    else:
        return 'Over-150'
    
# Define the desired order of categories
finish_category_order = [
    'Sub-60', '60-69',
    '70-79','80-89','90-99', '100-109', '110-119', '120-129',
    '130-139', '140-149',  'Over-150']
    
TOR330_itra_2018_2019_including_DNF_df  = TOR330_itra_2018_2019_including_DNF_df .rename(columns={"Performance": "Duration",
                                                    "Performance_Seconds": "Duration_seconds",
                                                    "ITRA_Nationality": "Nationality",
                                                   }) 
TOR330_itra_2018_2019_including_DNF_df ['Status'] = TOR330_itra_2018_2019_including_DNF_df ['Status'].str.replace('Finished', 'Finished at Courmayeur')
TOR330_itra_2018_2019_including_DNF_df ['Status1'] = TOR330_itra_2018_2019_including_DNF_df ['Status'].copy() 

TOR330_itra_2018_2019_including_DNF_df .loc[TOR330_itra_2018_2019_including_DNF_df ['Status1'] == 'Finished at Courmayeur', 'Status'] = 'True'
TOR330_itra_2018_2019_including_DNF_df .loc[TOR330_itra_2018_2019_including_DNF_df ['Status1'] == 'DNF', 'Status'] = 'False'

# Convert to timedelta and get total hours (handling NaT)
TOR330_itra_2018_2019_including_DNF_df ['Duration_hours'] = pd.to_timedelta(
    TOR330_itra_2018_2019_including_DNF_df ['Duration'], errors='coerce'
).dt.total_seconds() / 3600  # Convert seconds to hours



TOR330_itra_2018_2019_including_DNF_df ['Finish Category'] = TOR330_itra_2018_2019_including_DNF_df ['Duration_hours'].apply(categorize_duration)


# Set 'Finish Category' as a categorical column with the defined order
TOR330_itra_2018_2019_including_DNF_df ['Finish Category'] = pd.Categorical(
    TOR330_itra_2018_2019_including_DNF_df ['Finish Category'],
    categories = finish_category_order,
    ordered = True
)

In [51]:
def categorize_age(age):
    if age < 40: # under40
        return 'SEN'
    elif age < 50: #40-49
        return 'V1'
    elif age < 60: #50-59
        return 'V2'
    elif age < 70: #60-69
        return 'V3'
    elif age >= 70:
        return 'V4' # Over 70
    else:
        return '-'

TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Status1'] == 'True', 'Status1']  = 'Finished at Courmayeur'

TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Status1'] == 'False', 'Status1']  = 'DNFs'


In [52]:
TOR330_itra_2018_2019_including_DNF_df['Age'] = TOR330_itra_2018_2019_including_DNF_df['Age'].str.replace('-', '0')
TOR330_itra_2018_2019_including_DNF_df['Age'] = TOR330_itra_2018_2019_including_DNF_df['Age'].astype(int)

In [57]:
TOR330_itra_2018_2019_including_DNF_df ['Category'] = TOR330_itra_2018_2019_including_DNF_df['Age'].apply(categorize_age)

TOR330_itra_2018_2019_including_DNF_df  = TOR330_itra_2018_2019_including_DNF_df [[
    'Race','Year',  'Name',  'Sex', 'Nationality','Category',
       'Status', 'Status1',  'Duration_hours', 'Duration_seconds', 'Finish Category']]

TOR330_itra_2018_2019_including_DNF_df.head()

,Race,Year,Name,Sex,Nationality,Category,Status,Status1,Duration_hours,Duration_seconds,Finish Category
0,TOR330,2019,Bosatelli Oliviero,M,ITA,V2,True,Finished at Courmayeur,72.620278,261433.0,70-79
1,TOR330,2019,Reynolds Galen,M,CAN,SEN,True,Finished at Courmayeur,77.103333,277572.0,70-79
2,TOR330,2019,Lantermino Danilo,M,ITA,SEN,True,Finished at Courmayeur,79.162778,284986.0,70-79
3,TOR330,2019,Erwee Tiaan,M,RSA,SEN,True,Finished at Courmayeur,80.306667,289104.0,80-89
4,TOR330,2019,Lukas Jens,M,GER,V2,True,Finished at Courmayeur,85.067778,306244.0,80-89


In [61]:

# Mapping of 3-letter to 2-letter country codes
country_mapping = {
    ' ITA': 'IT', ' CAN': 'CA', ' RSA': 'ZA', ' GER': 'DE', ' ESP': 'ES', ' ROU': 'RO', ' FRA': 'FR', ' USA': 'US',
    ' GBR': 'GB', ' FIN': 'FI', ' CZE': 'CZ', ' CHN': 'CN', ' JPN': 'JP', ' CRO': 'HR', ' NOR': 'NO', ' BEL': 'BE',
    ' SUI': 'CH', ' COL': 'CO', ' AUS': 'AU', ' POR': 'PT', ' UKR': 'UA', ' HUN': 'HU', ' SLO': 'SI', ' SWE': 'SE',
    ' ISL': 'IS', ' BUL': 'BG', ' ARG': 'AR', ' MEX': 'MX', ' CRC': 'CR', ' NED': 'NL', ' IND': 'IN', ' NZL': 'NZ',
    ' DEN': 'DK', ' ECU': 'EC', ' BRA': 'BR', ' LTU': 'LT', ' TUR': 'TR', ' POL': 'PL', ' TPE': 'TW', ' SRB': 'RS',
    ' GRE': 'GR', ' AND': 'AD', ' SVK': 'SK', ' KOR': 'KR', ' RUS': 'RU', ' AUT': 'AT', ' MAS': 'MY', ' PHI': 'PH',
    ' SGP': 'SG', ' KEN': 'KE', ' HKG': 'HK', ' CHE': 'CH', ' MAC': 'MO', ' LBN': 'LB', ' THA': 'TH', ' INA': 'ID',
    ' MAD': 'MG', ' VIE': 'VN', ' IRL': 'IE', ' MDA': 'MD', ' CHI': 'CL', ' MON': 'MC', ' URU': 'UY', ' QAT': 'QA',
    ' PER': 'PE', ' MNE': 'ME', ' LUX': 'LU', ' ESA': 'SV', ' ISR': 'IL', ' BRU': 'BN', ' AFG': 'AF'
}

# Apply the mapping
TOR330_itra_2018_2019_including_DNF_df['Nationality'] = TOR330_itra_2018_2019_including_DNF_df['Nationality'].replace(country_mapping)

# Display updated DataFrame
print(TOR330_itra_2018_2019_including_DNF_df)


        Race  Year                Name Sex Nationality Category Status  \
0     TOR330  2019  Bosatelli Oliviero   M          IT       V2   True   
1     TOR330  2019      Reynolds Galen   M          CA      SEN   True   
2     TOR330  2019   Lantermino Danilo   M          IT      SEN   True   
3     TOR330  2019         Erwee Tiaan   M          ZA      SEN   True   
4     TOR330  2019          Lukas Jens   M          DE       V2   True   
...      ...   ...                 ...  ..         ...      ...    ...   
1815  TOR330  2018           Zdon Bill   M          US      SEN  False   
1816  TOR330  2018      Zennaro Davide   M          IT       V2  False   
1817  TOR330  2018        Zimei Andrea   M          IT      SEN  False   
1818  TOR330  2018   Zimmermann Denise   F          CH       V1  False   
1819  TOR330  2018        Zugna Davide   M          IT       V1  False   

                     Status1  Duration_hours  Duration_seconds Finish Category  
0     Finished at Courmayeur  

C:\Users\Karina\AppData\Local\Temp\ipykernel_12148\1947035403.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TOR330_itra_2018_2019_including_DNF_df['Nationality'] = TOR330_itra_2018_2019_including_DNF_df['Nationality'].replace(country_mapping)


In [62]:
for nationality in TOR330_itra_2018_2019_including_DNF_df['Nationality'].unique():
    if len(nationality) !=2:
        print(nationality)
#         print(f'TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df[\'Nationality\'] == \'{nationality}\', \'Nationality\'] == \'\'')


In [68]:
TOR330_dem['Year'] =TOR330_dem['Year'].astype(str)
TOR330_itra_2018_2019_including_DNF_df['Year'] =TOR330_itra_2018_2019_including_DNF_df['Year'].astype(str)

C:\Users\Karina\AppData\Local\Temp\ipykernel_12148\849963526.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TOR330_itra_2018_2019_including_DNF_df['Year'] =TOR330_itra_2018_2019_including_DNF_df['Year'].astype(str)


In [70]:
TOR330_itra_2018_2019_including_DNF_df['Year'].unique()

array(['2019', '2018'], dtype=object)

In [71]:
# Concatenate along columns (axis=1)
TOR330_dem_2018_2024 = pd.concat([TOR330_dem, TOR330_itra_2018_2019_including_DNF_df])

# Set 'Finish Category' as a categorical column with the defined order
TOR330_dem_2018_2024['Year'] = pd.Categorical(
    TOR330_dem_2018_2024['Year'],
    categories = ['2018','2019',  '2021', '2022', '2023', '2024'],
    ordered = True
)


TOR330_dem_2018_2024['Year'].unique()

['2021', '2022', '2023', '2024', '2019', '2018']
Categories (6, object): ['2018' < '2019' < '2021' < '2022' < '2023' < '2024']

In [72]:
n = 0

nationality_df = []
for name in list(TOR330_dem_2018_2024['Name'].unique()):
    name_df = TOR330_dem_2018_2024[TOR330_dem_2018_2024['Name'] == name]
    if len(list(name_df['Year'].unique())) == 1:
#         print(name_df['Name'].unique())
        n = n+1
#         print(n, name_df['Name'].unique(), name_df['Nationality'].unique())
        nationality_df.append(name_df)
    else:
        name_df.loc[name_df['Year'] == '2019', 'Nationality'] = np.nan
        name_df['Nationality'] = name_df['Nationality'].bfill()
        nationality_df.append(name_df)
        
TOR330_dem_2018_2024_1= pd.concat(nationality_df)

TOR330_dem_2018_2024_1[TOR330_dem_2018_2024_1['Name'] == 'Papi Luca'].reset_index(drop = True)

C:\Users\Karina\AppData\Local\Temp\ipykernel_12148\1754007779.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  name_df['Nationality'] = name_df['Nationality'].bfill()


['Jonas Russi']
1 ['Jonas Russi'] ['CH']
['Restorp Petter']
2 ['Restorp Petter'] ['FR']
['Lucas Jerome']
3 ['Lucas Jerome'] ['FR']
['Bethaz Marco']
4 ['Bethaz Marco'] ['IT']
['Bolis Matteo']
5 ['Bolis Matteo'] ['IT']
['Corneliu Buliga']
6 ['Corneliu Buliga'] ['RO']
['Laving Jean Marc']
7 ['Laving Jean Marc'] ['FR']
['Beaven Alexander']
8 ['Beaven Alexander'] ['GB']
['Christin Benoit Julien']
9 ['Christin Benoit Julien'] ['FR']
['Dunkerbeck Thomas']
10 ['Dunkerbeck Thomas'] ['NL']
['Roncato Alessandro']
11 ['Roncato Alessandro'] ['IT']
['Caffier Arthur']
12 ['Caffier Arthur'] ['ES']
['Poriel Yannick']
13 ['Poriel Yannick'] ['FR']
['Duclos Guillaume']
14 ['Duclos Guillaume'] ['FR']
['Tobias Bogner']
15 ['Tobias Bogner'] ['AT']
['Champigny Julien']
16 ['Champigny Julien'] ['FR']
['Hewitt Michael']
17 ['Hewitt Michael'] ['US']
['Orradre Inigo']
18 ['Orradre Inigo'] ['ES']
['Vieira Nelson']
19 ['Vieira Nelson'] ['PT']
['Poskin Julien']
20 ['Poskin Julien'] ['BE']
['Hetmanski Adam']
21 ['Het

['Soriano Guerrero Pedro']
168 ['Soriano Guerrero Pedro'] ['ES']
['Bouyer Fabien']
169 ['Bouyer Fabien'] ['FR']
['Banu Ovidiu']
170 ['Banu Ovidiu'] ['FR']
['Valverde Andres']
171 ['Valverde Andres'] ['ES']
['Vinaixa Ricard']
172 ['Vinaixa Ricard'] ['ES']
['Giannetti Stefano']
173 ['Giannetti Stefano'] ['IT']
['Corsini Valerio']
174 ['Corsini Valerio'] ['IT']
['Vincent Mengin']
175 ['Vincent Mengin'] ['FR']
['Veyres Dominique']
176 ['Veyres Dominique'] ['FR']
['Soulier Jerome']
177 ['Soulier Jerome'] ['FR']
['Peccavet Stephane']
178 ['Peccavet Stephane'] ['FR']
['Christophe Dietrich']
179 ['Christophe Dietrich'] ['FR']
['Galassi Cristina']
180 ['Galassi Cristina'] ['IT']
['Patrick Langelot']
181 ['Patrick Langelot'] ['FR']
['Lemaire Gwennael']
182 ['Lemaire Gwennael'] ['FR']
['Ingargiola Olivier']
183 ['Ingargiola Olivier'] ['FR']
['Thivolle Geraldine']
184 ['Thivolle Geraldine'] ['FR']
['Endstra Tom']
185 ['Endstra Tom'] ['NL']
['Ordas Enrique']
186 ['Ordas Enrique'] ['ES']
['Howard Ge

['Ciarla Alberta']
327 ['Ciarla Alberta'] ['IT']
['Chudy Przemyslaw']
328 ['Chudy Przemyslaw'] ['IE']
['Billy Laurent']
329 ['Billy Laurent'] ['FR']
['Schubert Sebastian']
330 ['Schubert Sebastian'] ['DE']
['Schuhmann Marc']
331 ['Schuhmann Marc'] ['FR']
['Serra Garcia Tomas']
332 ['Serra Garcia Tomas'] ['ES']
['Johnson Mark']
333 ['Johnson Mark'] ['US']
['Drouet Thierry']
334 ['Drouet Thierry'] ['FR']
['Boeretto Marco']
335 ['Boeretto Marco'] ['IT']
['Ranieri Fabrizio']
336 ['Ranieri Fabrizio'] ['IT']
['Stringari Guido']
337 ['Stringari Guido'] ['IT']
['Scarlatella Francesco']
338 ['Scarlatella Francesco'] ['IT']
['Fiorentini Francesco']
339 ['Fiorentini Francesco'] ['IT']
['Spenle Jean Martin']
340 ['Spenle Jean Martin'] ['FR']
['Sampaio Joaquim']
341 ['Sampaio Joaquim'] ['PT']
['Mayerhofer Thomas']
342 ['Mayerhofer Thomas'] ['AT']
['Ligasacchi Fabio']
343 ['Ligasacchi Fabio'] ['IT']
['Juang Than']
344 ['Juang Than'] ['TH']
['Banus Pedreny Delia']
345 ['Banus Pedreny Delia'] ['ES']
[

['Scagliotti Orlandini Luca']
500 ['Scagliotti Orlandini Luca'] ['IT']
['Giacchetta Stephane']
501 ['Giacchetta Stephane'] ['BE']
['Perez Comendador David']
502 ['Perez Comendador David'] ['ES']
['Ruiz Santiago Lucas']
503 ['Ruiz Santiago Lucas'] ['ES']
['Szkaluba Barbara']
504 ['Szkaluba Barbara'] ['GB']
['Lodi Severino']
505 ['Lodi Severino'] ['IT']
['Pella Liliana Beatriz']
506 ['Pella Liliana Beatriz'] ['IT']
['Bouquet Sophie']
507 ['Bouquet Sophie'] ['FR']
['Sudars Martins']
508 ['Sudars Martins'] ['LV']
['Byrne Simon Peter']
509 ['Byrne Simon Peter'] ['AU']
['Haylett Philip']
510 ['Haylett Philip'] ['GB']
['Luboz Dante']
511 ['Luboz Dante'] ['IT']
['Grise Dominic']
512 ['Grise Dominic'] ['GB']
['Augustson Ketil']
513 ['Augustson Ketil'] ['NO']
['Da Costa Veloso Alexandre']
514 ['Da Costa Veloso Alexandre'] ['AD']
['Le Duigou Sebastien']
515 ['Le Duigou Sebastien'] ['CH']
['Conta Paolo']
516 ['Conta Paolo'] ['IT']
['Beaumont Olivier']
517 ['Beaumont Olivier'] ['FR']
['Schwalger An

...

['Levai Valerie']
2666 ['Levai Valerie'] ['FR']
['Li Chao']
2667 ['Li Chao'] ['CN']
['Limontini Mattia']
2668 ['Limontini Mattia'] ['IT']
['Loeffen Dirk']
2669 ['Loeffen Dirk'] ['NL']
['Luglio Fabrizio']
2670 ['Luglio Fabrizio'] ['IT']
['Lyu Hao']
2671 ['Lyu Hao'] ['CN']
['Macchetto Nicola']
2672 ['Macchetto Nicola'] ['IT']
['Maneglia Andrea']
2673 ['Maneglia Andrea'] ['IT']
['Marechal Anthony']
2674 ['Marechal Anthony'] ['BE']
['Maresca Atilio']
2675 ['Maresca Atilio'] ['AR']
['Martin Frank']
2676 ['Martin Frank'] ['FR']
['Martines Luca']
2677 ['Martines Luca'] ['IT']
['Massart Jean Christophe']
2678 ['Massart Jean Christophe'] ['BE']
['Mattiolo Filippo']
2679 ['Mattiolo Filippo'] ['IT']
['Maurin Christophe']
2680 ['Maurin Christophe'] ['FR']
['Meazzi Riccardo']
2681 ['Meazzi Riccardo'] ['IT']
['Melacarne Michele']
2682 ['Melacarne Michele'] ['IT']
['Mercier Olivier']
2683 ['Mercier Olivier'] ['FR']
['Miao Lei']
2684 ['Miao Lei'] ['CN']
['Milazzo Angelo']
2685 ['Milazzo Angelo'] ['IT'

['Chaberge Fabrizio']
2842 ['Chaberge Fabrizio'] ['IT']
['Crippa Veronica']
2843 ['Crippa Veronica'] ['IT']
['Perrone Fodaro Carmelo']
2844 ['Perrone Fodaro Carmelo'] ['IT']
['Hajek Jakub']
2845 ['Hajek Jakub'] ['CZ']
['Hernandez Hernandez Narciso Ignacio']
2846 ['Hernandez Hernandez Narciso Ignacio'] ['ES']
['Javega David']
2847 ['Javega David'] ['ES']
['Moncanut Baptiste']
2848 ['Moncanut Baptiste'] ['FR']
['Oeillet Sylvain']
2849 ['Oeillet Sylvain'] ['FR']
['Calvo Requena Albert']
2850 ['Calvo Requena Albert'] ['AD']
['Betouret Sebastien']
2851 ['Betouret Sebastien'] ['FR']
['Olivson Oleksandr']
2852 ['Olivson Oleksandr'] ['UA']
['Cheng Chih Jen']
2853 ['Cheng Chih Jen'] ['TW']
['Zanarella Cristiano']
2854 ['Zanarella Cristiano'] ['IT']
['Lundstrom Jon']
2855 ['Lundstrom Jon'] ['DK']
['Carrara Ivano']
2856 ['Carrara Ivano'] ['IT']
['Pietrzak Krystian']
2857 ['Pietrzak Krystian'] ['PL']
['Bedhet Guillaume']
2858 ['Bedhet Guillaume'] ['FR']
['Aubert Florent']
2859 ['Aubert Florent'] [

['Herrera Noelia']
3032 ['Herrera Noelia'] ['ES']
['Guan Yadi']
3033 ['Guan Yadi'] ['CN']
['Ducly Remo']
3034 ['Ducly Remo'] ['IT']
['Javaheri Nima']
3035 ['Javaheri Nima'] ['GB']
['Komurcu Osman']
3036 ['Komurcu Osman'] ['GB']
['Marzano Massimo']
3037 ['Marzano Massimo'] ['IT']
['Delval Lionel']
3038 ['Delval Lionel'] ['FR']
['Lam Ka Man']
3039 ['Lam Ka Man'] ['HK']
['Ng Jonathan']
3040 ['Ng Jonathan'] ['HK']
['Vulcan Alessio']
3041 ['Vulcan Alessio'] ['IT']
['Cayuela Gomez Miguel Angel']
3042 ['Cayuela Gomez Miguel Angel'] ['ES']
['Martignoni Alessandro']
3043 ['Martignoni Alessandro'] ['IT']
['Sordi Paolo']
3044 ['Sordi Paolo'] ['IT']
['Fachino Sandro']
3045 ['Fachino Sandro'] ['IT']
['Sedda Ivan']
3046 ['Sedda Ivan'] ['IT']
['Arnold Andrew']
3047 ['Arnold Andrew'] ['GB']
['Xu Tianshu']
3048 ['Xu Tianshu'] ['CN']
['Garcia Marc']
3049 ['Garcia Marc'] ['ES']
['Capus Berard Isabelle']
3050 ['Capus Berard Isabelle'] ['FR']
['Priod Giancarlo']
3051 ['Priod Giancarlo'] ['IT']
['Medici And

['Glattstein Guillermo']
3206 ['Glattstein Guillermo'] ['AR']
['Goggi Rosanna']
3207 ['Goggi Rosanna'] ['IT']
['Goujon Jean Paul']
3208 ['Goujon Jean Paul'] ['FR']
['Haji Abdul Rahman Haji Abu Bakar']
3209 ['Haji Abdul Rahman Haji Abu Bakar'] ['BN']
['Hauser Ruckli Cornelia']
3210 ['Hauser Ruckli Cornelia'] ['CH']
['Heissl Thomas']
3211 ['Heissl Thomas'] ['AT']
['Heraud William']
3212 ['Heraud William'] ['FR']
['Herman Stephen']
3213 ['Herman Stephen'] ['BE']
['Holden Rob Holden']
3214 ['Holden Rob Holden'] ['FR']
['Holmes William']
3215 ['Holmes William'] ['US']
['Huishichen Huishichen']
3216 ['Huishichen Huishichen'] ['HK']
['Iancu David Traian']
3217 ['Iancu David Traian'] ['RO']
['Ilaria Rossetti']
3218 ['Ilaria Rossetti'] ['IT']
['Ireland Derek']
3219 ['Ireland Derek'] ['AU']
['Izzo Andrea']
3220 ['Izzo Andrea'] ['IT']
['Jaccond Ermando Ivan']
3221 ['Jaccond Ermando Ivan'] ['IT']
['Janiec Robert']
3222 ['Janiec Robert'] ['PL']
['Jarret Patrick']
3223 ['Jarret Patrick'] ['FR']
['Jo

,Year,Race,Name,Sex,Nationality,Category,Status,Status1,Duration_seconds,Finish Category,Duration_hours


In [78]:
TOR330_dem_2018_2024_1.groupby([ 'Year', 'Status1'])['Status1'].count()

Year  Status1                               
2018  DNF                                       345
      Finished at Bosses or Rifugio Frassati      0
      Finished at Courmayeur                    534
2019  DNF                                       378
      Finished at Bosses or Rifugio Frassati      0
      Finished at Courmayeur                    563
2021  DNF                                       286
      Finished at Bosses or Rifugio Frassati      0
      Finished at Courmayeur                    431
2022  DNF                                       363
      Finished at Bosses or Rifugio Frassati    189
      Finished at Courmayeur                    408
2023  DNF                                       585
      Finished at Bosses or Rifugio Frassati      0
      Finished at Courmayeur                    622
2024  DNF                                       563
      Finished at Bosses or Rifugio Frassati      0
      Finished at Courmayeur                    533
Name: Status1, dtyp

In [77]:
TOR330_dem_2018_2024_1['Status1'] =TOR330_dem_2018_2024_1['Status1'].str.replace('Finished at Bosses','Finished at Bosses or Rifugio Frassati') 
TOR330_dem_2018_2024_1['Status1'] =TOR330_dem_2018_2024_1['Status1'].str.replace('Finished at Rifugio Frassati','Finished at Bosses or Rifugio Frassati') 

In [80]:
TOR330_dem_2018_2024_1.to_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_dem_2018_2024.xlsx', index = False)